In [58]:
#inspired by: https://epjdatascience.springeropen.com/articles/10.1140/epjds/s13688-021-00265-y

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import geopandas as gpd
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


Inputs:
* SPEED_LIMI: int
* car: int
* bike: categorical (one_hot)
* curve_cut_: categorical
* dist_to_in: int

outputs: sev_simple: categorical (y)

In [43]:
data = gpd.read_file('prepped_roads.shp')
data.columns

Index(['CRASH_CRN', 'CRASH_YEAR', 'CRASH_MONT', 'DAY_OF_WEE', 'HOUR_OF_DA',
       'LOCATION_T', 'MAX_SEVERI', 'LONGITUDE', 'LATITUDE', 'SPEED_LIMI',
       'severity', 'carLane', 'bikeLane', 'car.revers', 'bike.rever',
       'foot_left', 'index_righ', 'Unnamed_ 0', 'edge_id', 'source', 'target',
       'car', 'car revers', 'bike', 'bike rever', 'foot_right', 'length2',
       'length_dir', 'dist_to_ro', 'dist_to_in', 'straight_l', 'curve',
       'curve_cut_', 'geometry'],
      dtype='object')

In [44]:
data = data[data['severity'] != 'unknown']
data['sev_simple'] = data['severity'].apply(lambda x: 0 if x == 'slight' else 1)
data['sev_simple'].value_counts()

0    312
1    264
Name: sev_simple, dtype: int64

In [45]:
data['curve_cat'] = data['curve_cut_'].astype('category')
data = pd.get_dummies(data, columns=['bike'])
data

,CRASH_CRN,CRASH_YEAR,CRASH_MONT,DAY_OF_WEE,HOUR_OF_DA,LOCATION_T,MAX_SEVERI,LONGITUDE,LATITUDE,SPEED_LIMI,...,dist_to_in,straight_l,curve,curve_cut_,geometry,sev_simple,curve_cat,bike_0,bike_2,bike_5
0,2004004909,2004,4,7,13.0,0,0,-79.906200,40.385600,25.0,...,4.458047,42.862069,1.000000,0,POINT (592840.061 4471130.694),0,0,0,1,0
1,2004016283,2004,4,5,15.0,0,3,-80.024600,40.408700,25.0,...,5.241868,42.466048,1.000000,0,POINT (582761.991 4473577.212),1,0,0,1,0
2,2004016780,2004,5,5,19.0,0,2,-79.903000,40.409100,35.0,...,17.132632,63.218544,1.039816,0,POINT (593079.306 4473742.586),1,0,0,1,0
3,2004019276,2004,1,3,23.0,0,1,-79.948100,40.418700,25.0,...,320.059565,1074.136221,1.001066,0,POINT (589239.862 4474761.673),1,0,0,1,0
4,2004078102,2004,9,2,10.0,0,4,-79.784800,40.450700,25.0,...,85.278335,380.071481,1.072430,0,POINT (603045.160 4478491.493),0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
848,2015114006,2015,11,4,7.0,0,4,-79.921501,40.468899,25.0,...,13.113883,84.623980,1.000000,0,POINT (591428.417 4480361.037),0,0,0,1,0
849,2015114934,2015,11,6,9.0,0,4,-79.967697,40.464500,25.0,...,6.693305,50.489964,1.000000,0,POINT (587517.850 4479825.986),0,0,0,1,0
850,2015119084,2015,11,2,8.0,0,3,-79.942299,40.440201,25.0,...,7.013571,12.392837,1.000000,0,POINT (589703.443 4477154.171),1,0,0,1,0
852,2015124094,2015,11,1,1.0,0,3,-79.949799,40.464199,25.0,...,3.615937,26.275600,1.000000,0,POINT (589035.701 4479810.433),1,0,0,1,0


In [36]:
data

Index(['CRASH_CRN', 'CRASH_YEAR', 'CRASH_MONT', 'DAY_OF_WEE', 'HOUR_OF_DA',
       'LOCATION_T', 'MAX_SEVERI', 'LONGITUDE', 'LATITUDE', 'SPEED_LIMI',
       'severity', 'carLane', 'bikeLane', 'car.revers', 'bike.rever',
       'foot_left', 'index_righ', 'Unnamed_ 0', 'edge_id', 'source', 'target',
       'car', 'car revers', 'bike rever', 'foot_right', 'length2',
       'length_dir', 'dist_to_ro', 'dist_to_in', 'straight_l', 'curve',
       'curve_cut_', 'geometry', 'speed_scale', 'curve_cat', 'bike_0',
       'bike_2', 'bike_5'],
      dtype='object')

In [50]:
data_cleaned = data.dropna()
data_cleaned

,CRASH_CRN,CRASH_YEAR,CRASH_MONT,DAY_OF_WEE,HOUR_OF_DA,LOCATION_T,MAX_SEVERI,LONGITUDE,LATITUDE,SPEED_LIMI,...,dist_to_in,straight_l,curve,curve_cut_,geometry,sev_simple,curve_cat,bike_0,bike_2,bike_5
0,2004004909,2004,4,7,13.0,0,0,-79.906200,40.385600,25.0,...,4.458047,42.862069,1.000000,0,POINT (592840.061 4471130.694),0,0,0,1,0
1,2004016283,2004,4,5,15.0,0,3,-80.024600,40.408700,25.0,...,5.241868,42.466048,1.000000,0,POINT (582761.991 4473577.212),1,0,0,1,0
2,2004016780,2004,5,5,19.0,0,2,-79.903000,40.409100,35.0,...,17.132632,63.218544,1.039816,0,POINT (593079.306 4473742.586),1,0,0,1,0
3,2004019276,2004,1,3,23.0,0,1,-79.948100,40.418700,25.0,...,320.059565,1074.136221,1.001066,0,POINT (589239.862 4474761.673),1,0,0,1,0
4,2004078102,2004,9,2,10.0,0,4,-79.784800,40.450700,25.0,...,85.278335,380.071481,1.072430,0,POINT (603045.160 4478491.493),0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
848,2015114006,2015,11,4,7.0,0,4,-79.921501,40.468899,25.0,...,13.113883,84.623980,1.000000,0,POINT (591428.417 4480361.037),0,0,0,1,0
849,2015114934,2015,11,6,9.0,0,4,-79.967697,40.464500,25.0,...,6.693305,50.489964,1.000000,0,POINT (587517.850 4479825.986),0,0,0,1,0
850,2015119084,2015,11,2,8.0,0,3,-79.942299,40.440201,25.0,...,7.013571,12.392837,1.000000,0,POINT (589703.443 4477154.171),1,0,0,1,0
852,2015124094,2015,11,1,1.0,0,3,-79.949799,40.464199,25.0,...,3.615937,26.275600,1.000000,0,POINT (589035.701 4479810.433),1,0,0,1,0


In [54]:
X = data_cleaned[['SPEED_LIMI', 'car', 'dist_to_in', 'curve_cat', 'bike_0', 'bike_2', 'bike_5']]
y = data_cleaned['sev_simple']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [55]:
model = LogisticRegression(solver='liblinear', random_state=42) # Choose a solver, e.g., 'liblinear', 'lbfgs'
model.fit(X_train, y_train)

LogisticRegression(random_state=42, solver='liblinear')

In [56]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test) # Get probabilities for each class

In [59]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5135135135135135

Confusion Matrix:
 [[52  5]
 [49  5]]

Classification Report:
               precision    recall  f1-score   support

           0       0.51      0.91      0.66        57
           1       0.50      0.09      0.16        54

    accuracy                           0.51       111
   macro avg       0.51      0.50      0.41       111
weighted avg       0.51      0.51      0.41       111



In [ ]:
#standardize?

In [25]:
scaler = StandardScaler()
data['speed_scale'] = scaler.fit_transform(data['SPEED_LIMI'].values.reshape(-1, 1))
data

,CRASH_CRN,CRASH_YEAR,CRASH_MONT,DAY_OF_WEE,HOUR_OF_DA,LOCATION_T,MAX_SEVERI,LONGITUDE,LATITUDE,SPEED_LIMI,...,foot_right,length2,length_dir,dist_to_ro,dist_to_in,straight_l,curve,curve_cut_,geometry,speed_scale
0,2004004909,2004,4,7,13.0,0,0,-79.906200,40.385600,25.0,...,1,0.000500,42.862069,2.335425,4.458047,42.862069,1.000000,0,POINT (592840.061 4471130.694),-0.360312
1,2004016283,2004,4,5,15.0,0,3,-80.024600,40.408700,25.0,...,1,0.000383,42.466049,2.713114,5.241868,42.466048,1.000000,0,POINT (582761.991 4473577.212),-0.360312
2,2004016780,2004,5,5,19.0,0,2,-79.903000,40.409100,35.0,...,1,0.000668,65.735639,5.042122,17.132632,63.218544,1.039816,0,POINT (593079.306 4473742.586),1.331498
3,2004019276,2004,1,3,23.0,0,1,-79.948100,40.418700,25.0,...,1,0.010272,1075.280734,0.053449,320.059565,1074.136221,1.001066,0,POINT (589239.862 4474761.673),-0.360312
4,2004078102,2004,9,2,10.0,0,4,-79.784800,40.450700,25.0,...,1,0.003898,407.600115,3.108009,85.278335,380.071481,1.072430,0,POINT (603045.160 4478491.493),-0.360312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
851,2015123541,2015,11,6,20.0,0,8,-79.840302,40.478298,35.0,...,1,0.002369,207.049767,3.383417,30.345828,207.049767,1.000000,0,POINT (598298.453 4481491.676),1.331498
852,2015124094,2015,11,1,1.0,0,3,-79.949799,40.464199,25.0,...,1,0.000274,26.275600,0.620458,3.615937,26.275600,1.000000,0,POINT (589035.701 4479810.433),-0.360312
853,2015127050,2015,12,2,7.0,0,8,-80.003502,40.452900,25.0,...,1,0.000181,19.839194,0.190372,0.640744,19.839194,1.000000,0,POINT (584496.835 4478503.429),-0.360312
854,2015127272,2015,12,2,8.0,0,8,-79.945602,40.444500,25.0,...,1,0.000027,2.973457,3.471096,3.471096,2.973457,1.000000,0,POINT (589417.569 4477628.035),-0.360312
